In [ ]:
# Đưa thư mục làm việc về gốc dự án (notebook nằm trong src/)
import os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
print("Thư mục làm việc:", os.getcwd())

## Bước 1: Import thư viện và kiểm tra bàn giao từ Notebook A

In [2]:
import os
print("CWD:", os.getcwd())
print("File tồn tại:", os.path.exists("data/processed/manifest.json"))

CWD: d:\ADYm
File tồn tại: True


In [3]:
import os
import json
import hashlib
import time
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from sklearn.metrics import (
    mean_absolute_error as MAE,
    mean_squared_error as MSE,
    f1_score,
    average_precision_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

# Tạo thư mục report nếu chưa có
os.makedirs("report", exist_ok=True)

with open("data/processed/manifest.json") as f:
    manifest = json.load(f)

with open("data/processed/feat.parquet", "rb") as f:
    current_md5 = hashlib.md5(f.read()).hexdigest()

# So MD5 với manifest — nếu khác, file đã bị sửa
assert current_md5 == manifest["md5"], (
    f"MD5 mismatch! Manifest ghi: {manifest['md5']}, "
    f"File hiện tại: {current_md5}. "
    f"Yêu cầu Sinh viên A chạy lại Notebook A."
)

feat = pd.read_parquet("data/processed/feat.parquet")

# Kiểm tra số dòng khớp manifest
assert len(feat) == manifest["rows"], (
    f"Số dòng lệch: manifest = {manifest['rows']}, "
    f"file thực = {len(feat)}"
)

con = duckdb.connect()
con.register("feat", feat)

def clean_axes_style(ax):
    """Loại đường viền top và right cho biểu đồ đẹp hơn."""
    ax.spines[["top", "right"]].set_visible(False)


print("Bàn giao OK")
print(f"  Số dòng     : {manifest['rows']:,}")
print(f"  Số cột      : {len(manifest['cols'])}")
print(f"  Thời gian   : {manifest['time_min']} → {manifest['time_max']}")
print(f"  Tác giả     : {manifest.get('author', 'N/A')}")

Bàn giao OK
  Số dòng     : 707,520
  Số cột      : 23
  Thời gian   : 2023-01-01 00:00:00 → 2026-09-01 23:00:00
  Tác giả     : Student A


## Bước 2: Chia dữ liệu train/test theo thời gian

In [4]:
feat = con.execute("SELECT * FROM feat").df()
feat["st_code"] = feat.station.astype("category").cat.codes
feat["ts"] = pd.to_datetime(feat["ts"])

feature_columns = [
    "sm2", "sm4", "sm8", "sm20",
    "sm8_lag1", "sm8_lag6", "sm8_lag24", "sm8_lag7d",
    "sm8_delta6", "sm8_delta24",
    "st2",
    "prcp", "r24", "r7d",
    "tair", "t3d",
    "ws",
    "hr", "mon",
    "st_code",
]

def prepare_data(horizon):
    """
    Chuẩn bị data cho 1 horizon (24 hoặc 72 giờ).
    Trả về dict chứa X_tr, X_te, y_tr_delta, y_te, sm8_te, naive_mae, ...
    """
    target_col = f"y_{horizon}h"
    
    required = feature_columns + [target_col, "sm8_lag7d", "sm8"]
    df = feat.dropna(subset=required).reset_index(drop=True)
    
    train = df[df.ts < "2025-12-31"]
    test  = df[df.ts >= "2026-01-01"]
    
    X_tr = train[feature_columns].astype("float32").values
    y_tr_raw = train[target_col].astype("float32").values
    sm8_tr = train["sm8"].astype("float32").values
    
    X_te = test[feature_columns].astype("float32").values
    y_te = test[target_col].astype("float32").values
    sm8_te = test["sm8"].astype("float32").values
    
    y_tr_delta = y_tr_raw - sm8_tr
    naive_prediction = sm8_te
    naive_mae = MAE(y_te, naive_prediction)
    
    print(f"[{horizon}h] Train: {len(train):,} dòng | Test: {len(test):,} dòng | Naive MAE: {naive_mae:.3f}")
    
    return {
        "horizon": horizon,
        "train": train, "test": test,
        "X_tr": X_tr, "X_te": X_te,
        "y_tr_delta": y_tr_delta, "y_te": y_te,
        "sm8_tr": sm8_tr, "sm8_te": sm8_te,
        "naive_prediction": naive_prediction,
        "naive_mae": naive_mae,
    }

# Chuẩn bị data cho horizon 24h (mặc định - dùng cho các phân tích chính)
data_24 = prepare_data(24)

# Xuất biến 24h ra để các CELL sau (SHAP, tuning, transfer...) dùng
train, test = data_24["train"], data_24["test"]
X_tr, X_te = data_24["X_tr"], data_24["X_te"]
y_tr_delta, y_te = data_24["y_tr_delta"], data_24["y_te"]
sm8_tr, sm8_te = data_24["sm8_tr"], data_24["sm8_te"]
naive_prediction, naive_mae = data_24["naive_prediction"], data_24["naive_mae"]

# Lưu feature table
def classify_feature_group(feature_name):
    if any(k in feature_name for k in ["lag", "r24", "r7d", "t3d", "delta"]):
        return "lag/rolling/delta"
    if feature_name in ("hr", "mon"):
        return "calendar"
    return "exogenous/structural"

feature_table = pd.DataFrame({
    "feature": feature_columns,
    "group":   [classify_feature_group(c) for c in feature_columns],
})
feature_table.to_csv("report/table_features.csv", index=False)

assert len(X_te) > max(3000, 0.1 * len(X_tr)), f"Test set quá nhỏ ({len(X_te)})"

[24h] Train: 397,882 dòng | Test: 95,030 dòng | Naive MAE: 0.629


## Bước 3a: Huấn luyện 5 mô hình × 5 seed (RQ2)

In [5]:
import os 
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping


def make_ridge(seed):
    return Ridge(alpha=1.0)

def make_random_forest(seed):
    return RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=seed)

def make_lightgbm(seed):
    return LGBMRegressor(
        n_estimators=800, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        random_state=seed, verbose=-1,
    )

def make_xgboost(seed):
    return XGBRegressor(
        n_estimators=800, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=seed,
    )

# ============================================================
# HÀM CHUẨN BỊ SEQUENCE CHO LSTM
# ============================================================
LOOKBACK = 24

def make_sequences(X, y_delta, sm8_now, lookback=LOOKBACK):
    """Reshape (samples, features) → (samples, timesteps, features) cho LSTM."""
    n = len(X)
    X_seq = np.zeros((n - lookback, lookback, X.shape[1]), dtype=np.float32)
    for i in range(lookback, n):
        X_seq[i - lookback] = X[i - lookback:i]
    y_seq = y_delta[lookback:]
    sm8_seq = sm8_now[lookback:]
    return X_seq, y_seq, sm8_seq


def make_lstm_model(input_shape):
    """LSTM 2 layers."""
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mae',
    )
    return model


# ============================================================
# HÀM CHÍNH: TRAIN + ĐÁNH GIÁ TẤT CẢ MODEL (bao gồm LSTM)
# ============================================================
def train_and_evaluate(data, n_seeds=5, n_seeds_lstm=3):
    """
    Train 4 model tabular + LSTM trên data. Trả về bảng kết quả đầy đủ.
    """
    horizon = data["horizon"]
    X_tr, y_tr_delta = data["X_tr"], data["y_tr_delta"]
    X_te, y_te = data["X_te"], data["y_te"]
    sm8_tr, sm8_te = data["sm8_tr"], data["sm8_te"]
    naive_mae = data["naive_mae"]
    naive_prediction = data["naive_prediction"]

    rows = []

    # ---- 4 model tabular ----
    tabular_models = {
        "Ridge":         make_ridge,
        "Random Forest": make_random_forest,
        "LightGBM":      make_lightgbm,
        "XGBoost":       make_xgboost,
    }

    for model_name, make_model_fn in tabular_models.items():
        mae_scores, rmse_scores, train_times = [], [], []
        print(f"[{horizon}h] Đang chạy {model_name}...", end=" ")

        for seed in range(n_seeds):
            model = make_model_fn(seed)
            start = time.perf_counter()
            model.fit(X_tr, y_tr_delta)
            train_times.append(time.perf_counter() - start)

            prediction = model.predict(X_te) + sm8_te
            mae_scores.append(MAE(y_te, prediction))
            rmse_scores.append(MSE(y_te, prediction) ** 0.5)

        rows.append([
            model_name,
            np.mean(mae_scores), np.std(mae_scores),
            np.mean(rmse_scores), np.std(rmse_scores),
            np.mean(train_times),
        ])
        print(f"MAE = {np.mean(mae_scores):.3f} ± {np.std(mae_scores):.3f}")

    # ---- LSTM (cần data dạng sequence) ----
    print(f"\n[{horizon}h] Đang chuẩn bị sequence cho LSTM...")
    X_tr_seq, y_tr_seq, _ = make_sequences(X_tr, y_tr_delta, sm8_tr)
    X_te_seq, _, sm8_te_seq = make_sequences(X_te, y_te - sm8_te, sm8_te)
    y_te_seq_true = y_te[LOOKBACK:]
    
    print(f"  X_tr_seq shape: {X_tr_seq.shape}, X_te_seq shape: {X_te_seq.shape}")
    
    mae_scores_lstm, rmse_scores_lstm, train_times_lstm = [], [], []

    for seed in range(n_seeds_lstm):
        print(f"[{horizon}h] Đang chạy LSTM seed={seed}...")
        tf.random.set_seed(seed)
        np.random.seed(seed)

        model_lstm = make_lstm_model((LOOKBACK, X_tr.shape[1]))
        start = time.perf_counter()
        model_lstm.fit(
            X_tr_seq, y_tr_seq,
            epochs=10,
            batch_size=512,
            validation_split=0.1,
            callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
            verbose=0,
        )
        train_times_lstm.append(time.perf_counter() - start)

        pred_delta = model_lstm.predict(X_te_seq, batch_size=512, verbose=0).flatten()
        prediction_lstm = pred_delta + sm8_te_seq
        mae_scores_lstm.append(MAE(y_te_seq_true, prediction_lstm))
        rmse_scores_lstm.append(MSE(y_te_seq_true, prediction_lstm) ** 0.5)

    print(f"[{horizon}h] LSTM MAE = {np.mean(mae_scores_lstm):.3f} ± {np.std(mae_scores_lstm):.3f}")
    
    rows.append([
        "LSTM",
        np.mean(mae_scores_lstm), np.std(mae_scores_lstm),
        np.mean(rmse_scores_lstm), np.std(rmse_scores_lstm),
        np.mean(train_times_lstm),
    ])

    # ---- Ghép bảng ----
    df = pd.DataFrame(rows, columns=["model", "MAE", "sd_MAE", "RMSE", "sd_RMSE", "train_seconds"])

    naive_row = pd.DataFrame([[
        "Naive (baseline)",
        naive_mae, 0.0,
        MSE(y_te, naive_prediction) ** 0.5, 0.0,
        0.0,
    ]], columns=df.columns)

    full = pd.concat([naive_row, df], ignore_index=True)
    full["improvement_vs_naive_%"] = (1 - full["MAE"] / naive_mae) * 100
    return full, df


# ============================================================
# CHẠY CHO HORIZON 24H
# ============================================================
print("=" * 70)
print("HORIZON 24H")
print("=" * 70)
result_full, result_df = train_and_evaluate(data_24)
result_full.round(3).to_csv("report/table_rq2.csv", index=False)
print("\n" + result_full.round(3).to_string(index=False))

# Xếp hạng
non_naive = result_df.copy().sort_values("MAE").reset_index(drop=True)
print("\n=== XẾP HẠNG (24h) ===")
for i, row in non_naive.iterrows():
    marker = "🥇" if i == 0 else "🥈" if i == 1 else "🥉" if i == 2 else "  "
    print(f"{marker} {i+1}. {row['model']:20s} MAE = {row['MAE']:.3f}")

best_model_name = non_naive.iloc[0]["model"]
print(f"\nMô hình tốt nhất (24h): {best_model_name}")


HORIZON 24H
[24h] Đang chạy Ridge... MAE = 0.654 ± 0.000
[24h] Đang chạy Random Forest... MAE = 0.830 ± 0.004
[24h] Đang chạy LightGBM... MAE = 0.669 ± 0.008
[24h] Đang chạy XGBoost... MAE = 0.715 ± 0.005

[24h] Đang chuẩn bị sequence cho LSTM...
  X_tr_seq shape: (397858, 24, 20), X_te_seq shape: (95006, 24, 20)
[24h] Đang chạy LSTM seed=0...


[24h] Đang chạy LSTM seed=1...
[24h] Đang chạy LSTM seed=2...
[24h] LSTM MAE = 0.543 ± 0.004

           model   MAE  sd_MAE  RMSE  sd_RMSE  train_seconds  improvement_vs_naive_%
Naive (baseline) 0.629   0.000 2.002    0.000          0.000                   0.000
           Ridge 0.654   0.000 1.933    0.000          0.347                  -4.029
   Random Forest 0.830   0.004 2.102    0.004        329.285                 -32.001
        LightGBM 0.669   0.008 1.907    0.009          4.555                  -6.408
         XGBoost 0.715   0.005 1.945    0.009          8.944                 -13.674
            LSTM 0.543   0.004 1.980    0.002  

In [6]:
# --- Chạy cho horizon 72h ---
print("\n" + "=" * 70)
print("HORIZON 72H")
print("=" * 70)

data_72 = prepare_data(72)
result_full_72, result_df_72 = train_and_evaluate(data_72)
result_full_72.round(3).to_csv("report/table_rq2_72h.csv", index=False)
print(result_full_72.round(3))

# --- Bảng so sánh 2 horizon ---
comparison = pd.DataFrame({
    "model":            result_full["model"],
    "MAE_24h":          result_full["MAE"].round(3),
    "improvement_24h_%": result_full["improvement_vs_naive_%"].round(1),
    "MAE_72h":          result_full_72["MAE"].round(3),
    "improvement_72h_%": result_full_72["improvement_vs_naive_%"].round(1),
})

comparison.to_csv("report/table_horizon_comparison.csv", index=False)

print("\n" + "=" * 70)
print("SO SÁNH 24H vs 72H")
print("=" * 70)
print(comparison.to_string(index=False))

best_24_imp = result_full.iloc[1:]["improvement_vs_naive_%"].max()
best_72_imp = result_full_72.iloc[1:]["improvement_vs_naive_%"].max()

print(f"\n• Horizon 24h — cải thiện tối đa: {best_24_imp:.1f}%")
print(f"• Horizon 72h — cải thiện tối đa: {best_72_imp:.1f}%")

if best_72_imp > best_24_imp + 10:
    print("\n→ ML thắng persistence RÕ RỆT ở horizon dài (72h), yếu ở horizon ngắn (24h).")
    print("   Phù hợp với ứng dụng CẢNH BÁO HẠN NHIỀU NGÀY.")
elif best_72_imp > best_24_imp:
    print("\n→ ML có ưu thế TỐT HƠN ở horizon dài, đúng như kỳ vọng.")
else:
    print("\n→ ML không có ưu thế đáng kể ở cả 2 horizon.")

# --- Biểu đồ so sánh ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, res, title in [
    (axes[0], result_full,    "Horizon 24h"),
    (axes[1], result_full_72, "Horizon 72h"),
]:
    mae_vals = res["MAE"].values
    lowest = mae_vals[1:].min()
    colors = ["#F4A261" if v == lowest else "#1B6B6D" for v in mae_vals]
    bars = ax.bar(res["model"], mae_vals, color=colors)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in mae_vals],
                 fontweight="bold", fontsize=9)
    ax.set_ylabel("MAE")
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)
    clean_axes_style(ax)

fig.tight_layout()
fig.savefig("report/fig_horizon_comparison.png", dpi=300)
plt.close(fig)
print("\nĐã lưu: report/fig_horizon_comparison.png")


HORIZON 72H
[72h] Train: 397,279 dòng | Test: 93,798 dòng | Naive MAE: 1.381
[72h] Đang chạy Ridge... MAE = 1.436 ± 0.000
[72h] Đang chạy Random Forest... MAE = 1.985 ± 0.006
[72h] Đang chạy LightGBM... MAE = 1.526 ± 0.009
[72h] Đang chạy XGBoost... MAE = 1.581 ± 0.013

[72h] Đang chuẩn bị sequence cho LSTM...
  X_tr_seq shape: (397255, 24, 20), X_te_seq shape: (93774, 24, 20)
[72h] Đang chạy LSTM seed=0...
[72h] Đang chạy LSTM seed=1...
[72h] Đang chạy LSTM seed=2...
[72h] LSTM MAE = 1.224 ± 0.008
              model    MAE  sd_MAE   RMSE  sd_RMSE  train_seconds  \
0  Naive (baseline)  1.381   0.000  3.169    0.000          0.000   
1             Ridge  1.436   0.000  3.022    0.000          0.303   
2     Random Forest  1.985   0.006  3.750    0.012        128.218   
3          LightGBM  1.526   0.009  3.062    0.011          3.364   
4           XGBoost  1.581   0.013  3.088    0.025          5.538   
5              LSTM  1.224   0.008  3.125    0.001        291.093   

   improvem

In [7]:
# ============================================================
# STATISTICAL SIGNIFICANCE TEST - HORIZON 24H
# ============================================================
from scipy.stats import ttest_rel, norm
import numpy as np

print("=" * 70)
print("STATISTICAL SIGNIFICANCE TEST — HORIZON 24H")
print("=" * 70)

# Lấy data H24 (đảm bảo dùng đúng)
X_tr_24 = data_24["X_tr"]
X_te_24 = data_24["X_te"]
y_tr_delta_24 = data_24["y_tr_delta"]
y_te_24 = data_24["y_te"]
sm8_tr_24 = data_24["sm8_tr"]
sm8_te_24 = data_24["sm8_te"]
pred_naive_24 = data_24["naive_prediction"]

# Train Ridge H24
from sklearn.linear_model import Ridge
m_ridge = Ridge(alpha=1.0).fit(X_tr_24, y_tr_delta_24)
pred_ridge_24 = m_ridge.predict(X_te_24) + sm8_te_24

# Train LightGBM H24
from lightgbm import LGBMRegressor
m_lgb = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31,
                      subsample=0.8, colsample_bytree=0.8,
                      random_state=0, verbose=-1).fit(X_tr_24, y_tr_delta_24)
pred_lgb_24 = m_lgb.predict(X_te_24) + sm8_te_24

# Train LSTM H24
import tensorflow as tf
tf.random.set_seed(0)
np.random.seed(0)

X_tr_seq_24, y_tr_seq_24, _ = make_sequences(X_tr_24, y_tr_delta_24, sm8_tr_24)
X_te_seq_24, _, sm8_te_seq_24 = make_sequences(X_te_24, y_te_24 - sm8_te_24, sm8_te_24)
y_te_seq_true_24 = y_te_24[LOOKBACK:]

m_lstm = make_lstm_model((LOOKBACK, X_tr_24.shape[1]))
m_lstm.fit(X_tr_seq_24, y_tr_seq_24, epochs=10, batch_size=512,
           validation_split=0.1, verbose=0,
           callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
pred_lstm_24 = m_lstm.predict(X_te_seq_24, batch_size=512, verbose=0).flatten() + sm8_te_seq_24

# Absolute errors
err_naive_24 = np.abs(y_te_24 - pred_naive_24)
err_ridge_24 = np.abs(y_te_24 - pred_ridge_24)
err_lgb_24 = np.abs(y_te_24 - pred_lgb_24)
err_lstm_24 = np.abs(y_te_seq_true_24 - pred_lstm_24)

# Align cho LSTM (bỏ 24 samples đầu)
err_naive_24_aligned = err_naive_24[LOOKBACK:]
err_ridge_24_aligned = err_ridge_24[LOOKBACK:]
err_lgb_24_aligned = err_lgb_24[LOOKBACK:]

# Paired t-tests
print("\n--- Paired t-tests ---")
comparisons_24 = [
    ("LSTM vs Naive",  err_lstm_24, err_naive_24_aligned),
    ("LSTM vs Ridge",  err_lstm_24, err_ridge_24_aligned),
    ("LSTM vs LGBM",   err_lstm_24, err_lgb_24_aligned),
    ("Ridge vs Naive", err_ridge_24, err_naive_24),
    ("LGBM vs Naive",  err_lgb_24, err_naive_24),
]

results_24 = []
for name, err_a, err_b in comparisons_24:
    t, p = ttest_rel(err_a, err_b)
    sig = "*** significant" if p < 0.001 else ("** p<0.01" if p < 0.01 else ("* p<0.05" if p < 0.05 else "not sig"))
    direction = "A better" if t < 0 else "A worse"
    print(f"{name:20s}: t = {t:>8.2f}, p = {p:.2e}, {sig}, {direction}")
    results_24.append([name, t, p, sig])

# Lưu results
import pandas as pd
pd.DataFrame(results_24, columns=["comparison", "t_statistic", "p_value", "significance"]).to_csv(
    "report/table_stat_test_h24.csv", index=False)
print("\nĐã lưu: report/table_stat_test_h24.csv")

STATISTICAL SIGNIFICANCE TEST — HORIZON 24H

--- Paired t-tests ---
LSTM vs Naive       : t =   -74.36, p = 0.00e+00, *** significant, A better
LSTM vs Ridge       : t =   -73.60, p = 0.00e+00, *** significant, A better
LSTM vs LGBM        : t =   -55.49, p = 0.00e+00, *** significant, A better
Ridge vs Naive      : t =    20.04, p = 3.83e-89, *** significant, A worse
LGBM vs Naive       : t =    20.91, p = 7.42e-97, *** significant, A worse

Đã lưu: report/table_stat_test_h24.csv


In [8]:
# ============================================================
# STATISTICAL SIGNIFICANCE TEST - HORIZON 72H
# ============================================================

print("=" * 70)
print("STATISTICAL SIGNIFICANCE TEST — HORIZON 72H")
print("=" * 70)

# Lấy data H72
X_tr_72 = data_72["X_tr"]
X_te_72 = data_72["X_te"]
y_tr_delta_72 = data_72["y_tr_delta"]
y_te_72 = data_72["y_te"]
sm8_tr_72 = data_72["sm8_tr"]
sm8_te_72 = data_72["sm8_te"]
pred_naive_72 = data_72["naive_prediction"]

# Train models H72
m_ridge_72 = Ridge(alpha=1.0).fit(X_tr_72, y_tr_delta_72)
pred_ridge_72 = m_ridge_72.predict(X_te_72) + sm8_te_72

m_lgb_72 = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=0, verbose=-1).fit(X_tr_72, y_tr_delta_72)
pred_lgb_72 = m_lgb_72.predict(X_te_72) + sm8_te_72

tf.random.set_seed(0)
np.random.seed(0)

X_tr_seq_72, y_tr_seq_72, _ = make_sequences(X_tr_72, y_tr_delta_72, sm8_tr_72)
X_te_seq_72, _, sm8_te_seq_72 = make_sequences(X_te_72, y_te_72 - sm8_te_72, sm8_te_72)
y_te_seq_true_72 = y_te_72[LOOKBACK:]

m_lstm_72 = make_lstm_model((LOOKBACK, X_tr_72.shape[1]))
m_lstm_72.fit(X_tr_seq_72, y_tr_seq_72, epochs=10, batch_size=512,
              validation_split=0.1, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
pred_lstm_72 = m_lstm_72.predict(X_te_seq_72, batch_size=512, verbose=0).flatten() + sm8_te_seq_72

# Absolute errors
err_naive_72 = np.abs(y_te_72 - pred_naive_72)
err_ridge_72 = np.abs(y_te_72 - pred_ridge_72)
err_lgb_72 = np.abs(y_te_72 - pred_lgb_72)
err_lstm_72 = np.abs(y_te_seq_true_72 - pred_lstm_72)

err_naive_72_aligned = err_naive_72[LOOKBACK:]
err_ridge_72_aligned = err_ridge_72[LOOKBACK:]
err_lgb_72_aligned = err_lgb_72[LOOKBACK:]

# Paired t-tests
print("\n--- Paired t-tests ---")
comparisons_72 = [
    ("LSTM vs Naive",  err_lstm_72, err_naive_72_aligned),
    ("LSTM vs Ridge",  err_lstm_72, err_ridge_72_aligned),
    ("LSTM vs LGBM",   err_lstm_72, err_lgb_72_aligned),
    ("Ridge vs Naive", err_ridge_72, err_naive_72),
    ("LGBM vs Naive",  err_lgb_72, err_naive_72),
]

results_72 = []
for name, err_a, err_b in comparisons_72:
    t, p = ttest_rel(err_a, err_b)
    sig = "*** significant" if p < 0.001 else ("** p<0.01" if p < 0.01 else ("* p<0.05" if p < 0.05 else "not sig"))
    direction = "A better" if t < 0 else "A worse"
    print(f"{name:20s}: t = {t:>8.2f}, p = {p:.2e}, {sig}, {direction}")
    results_72.append([name, t, p, sig])

pd.DataFrame(results_72, columns=["comparison", "t_statistic", "p_value", "significance"]).to_csv(
    "report/table_stat_test_h72.csv", index=False)
print("\nĐã lưu: report/table_stat_test_h72.csv")

# ============================================================
# TỔNG KẾT
# ============================================================
print("\n" + "=" * 70)
print("TỔNG KẾT")
print("=" * 70)
print(f"\nH24: LSTM MAE = {err_lstm_24.mean():.3f}, Naive MAE = {err_naive_24_aligned.mean():.3f}")
print(f"H72: LSTM MAE = {err_lstm_72.mean():.3f}, Naive MAE = {err_naive_72_aligned.mean():.3f}")

STATISTICAL SIGNIFICANCE TEST — HORIZON 72H

--- Paired t-tests ---
LSTM vs Naive       : t =   -51.58, p = 0.00e+00, *** significant, A better
LSTM vs Ridge       : t =   -70.66, p = 0.00e+00, *** significant, A better
LSTM vs LGBM        : t =   -56.49, p = 0.00e+00, *** significant, A better
Ridge vs Naive      : t =    25.57, p = 9.61e-144, *** significant, A worse
LGBM vs Naive       : t =    25.98, p = 2.70e-148, *** significant, A worse

Đã lưu: report/table_stat_test_h72.csv

TỔNG KẾT

H24: LSTM MAE = 0.544, Naive MAE = 0.629
H72: LSTM MAE = 1.231, Naive MAE = 1.381


### 3a.1. Bốn biểu đồ trực quan hoá kết quả RQ2

In [9]:
# Huấn luyện LightGBM chính (seed 0) để lấy dự báo cho các biểu đồ và SHAP
best_model = make_lightgbm(seed=0).fit(X_tr, y_tr_delta)   # ← DELTA
prediction = best_model.predict(X_te) + sm8_te             # ← khôi phục về level

print(f"MAE LightGBM (seed 0): {MAE(y_te, prediction):.3f}")

MAE LightGBM (seed 0): 0.682


In [10]:
fig, ax = plt.subplots(figsize=(7, 3.2))

mae_values = result_full["MAE"].values
model_names = result_full["model"].values

# Tìm MAE thấp nhất trong các mô hình (bỏ naive - dòng 0)
lowest_non_naive_mae = mae_values[1:].min()

# Màu: cam cho mô hình tốt nhất, teal cho còn lại
colors = [
    "#F4A261" if v == lowest_non_naive_mae else "#1B6B6D"
    for v in mae_values
]

bars = ax.bar(model_names, mae_values, color=colors)

# Ghi nhãn MAE bên trên mỗi cột
ax.bar_label(
    bars,
    labels=[f"{v:.2f}" for v in mae_values],
    fontweight="bold",
    fontsize=9,
)

ax.set_ylabel("MAE")
clean_axes_style(ax)
plt.xticks(rotation=15)

fig.tight_layout()
fig.savefig("report/fig_rq2_models.png", dpi=300)
plt.close(fig)
print("Đã lưu: report/fig_rq2_models.png")

Đã lưu: report/fig_rq2_models.png


In [11]:
# Ghép cột dự báo vào bảng test
test_with_prediction = test.assign(pred=prediction)

# Chọn 1 trạm để vẽ (trạm đầu tiên)
sample_station = test_with_prediction["station"].iloc[0]
station_data = (
    test_with_prediction[test_with_prediction["station"] == sample_station]
    .sort_values("ts")
)

fig, ax = plt.subplots(figsize=(9, 3))

ax.plot(
    station_data["ts"], station_data["y_24h"],
    label="Actual", color="#0F3D3E",
)
ax.plot(
    station_data["ts"], station_data["pred"],
    label="LightGBM", color="#F4A261",
)

ax.set_ylabel("sm8 (%)")
ax.legend(frameon=False)
clean_axes_style(ax)

fig.tight_layout()
fig.savefig("report/fig_rq2_pred_vs_actual.png", dpi=300)
plt.close(fig)
print(f"Đã lưu: report/fig_rq2_pred_vs_actual.png (trạm {sample_station})")

Đã lưu: report/fig_rq2_pred_vs_actual.png (trạm Levelland (2105))


In [12]:
residuals = y_te - prediction

fig, axes = plt.subplots(1, 2, figsize=(9, 3))

# Subplot trái: Histogram residual
axes[0].hist(residuals, bins=50, color="#1B6B6D")
axes[0].set_xlabel("Residual (Actual - Predicted)")
axes[0].axvline(x=0, color="#F4A261", linewidth=1)   # đường 0 để dễ nhìn bias
clean_axes_style(axes[0])

# Subplot phải: Scatter predicted vs actual
axes[1].scatter(prediction, y_te, s=4, alpha=0.3, color="#1B6B6D")
axes[1].plot(
    [y_te.min(), y_te.max()],
    [y_te.min(), y_te.max()],
    color="#F4A261", linewidth=1,
)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
clean_axes_style(axes[1])

fig.tight_layout()
fig.savefig("report/fig_rq2_residual.png", dpi=300)
plt.close(fig)
print("Đã lưu: report/fig_rq2_residual.png")

Đã lưu: report/fig_rq2_residual.png


In [ ]:
error_by_station = (
    test_with_prediction
    .groupby("station")
    .apply(lambda g: MAE(g["y_24h"], g["pred"]))
    .sort_values()
)

# Lưu toàn bộ (không chỉ 20 trạm) ra CSV
error_by_station.round(3).to_csv("report/table_rq2_error_by_unit.csv")

# Vẽ 20 trạm thấp nhất
fig, ax = plt.subplots(figsize=(9, 3))
error_by_station.head(20).plot.bar(ax=ax, color="#1B6B6D")
ax.set_ylabel("MAE")
ax.set_xlabel("Trạm")
clean_axes_style(ax)

fig.tight_layout()
fig.savefig("report/fig_rq2_error_by_unit.png", dpi=300)
plt.close(fig)

# In top 3 trạm tốt nhất và 3 trạm tệ nhất
print("3 trạm có MAE THẤP nhất:", error_by_station.head(3).round(2).to_dict())
print("3 trạm có MAE CAO nhất :", error_by_station.tail(3).round(2).to_dict())

3 trạm có MAE THẤP nhất: {'Levelland (2105)': 0.17, 'Lehman (2106)': 0.23, 'Reese Center (2104)': 0.26}
3 trạm có MAE CAO nhất : {'Ku-Nesa (2147)': 1.13, 'Abrams (2092)': 1.81, 'Vernon (2202)': 2.22}


### 3a.2. Tinh chỉnh hyperparameter cho LightGBM

In [14]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import loguniform, randint, uniform

parameter_space = {
    "learning_rate":     loguniform(0.01, 0.1),
    "num_leaves":        randint(15, 64),
    "n_estimators":      randint(300, 1500),
    "subsample":         uniform(0.6, 0.4),
    "colsample_bytree":  uniform(0.6, 0.4),
    "min_child_samples": randint(10, 100),
}

search = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=0, verbose=-1),
    param_distributions=parameter_space,
    n_iter=30,                                
    cv=TimeSeriesSplit(5),                    
    scoring="neg_mean_absolute_error",        
    random_state=42,
    n_jobs=-1,                               
)

print("Đang tinh chỉnh (30 tổ hợp × 5 fold = 150 lần huấn luyện)...")
search.fit(X_tr, y_tr_delta)   # ← DELTA

tuned_model = search.best_estimator_
tuned_prediction = tuned_model.predict(X_te) + sm8_te   # ← khôi phục

default_mae = MAE(y_te, prediction)
tuned_mae   = MAE(y_te, tuned_prediction)

tuning_table = pd.DataFrame([
    ["LightGBM default", default_mae, MSE(y_te, prediction) ** 0.5],
    ["LightGBM tuned",   tuned_mae,   MSE(y_te, tuned_prediction) ** 0.5],
], columns=["model", "MAE", "RMSE"])

tuning_table.round(3).to_csv("report/table_tuning.csv", index=False)
print("\n" + tuning_table.round(3).to_string(index=False))
print("\nTham số tối ưu tìm được:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

seed_std = result_df.loc[result_df["model"] == "LightGBM", "sd_MAE"].values[0]
mae_improvement = default_mae - tuned_mae

print(f"\nCải thiện MAE  : {mae_improvement:.4f}")
print(f"Std qua 5 seed  : {seed_std:.4f}")

if mae_improvement > seed_std:
    print("→ TINH CHỈNH ĐÁNG BÁO CÁO (giảm MAE nhiều hơn nhiễu seed)")
else:
    print("→ TINH CHỈNH KHÔNG CẦN THIẾT (cải thiện < nhiễu seed)")

with open("report/params_tuned.json", "w") as f:
    json.dump(search.best_params_, f, indent=2, default=float)

Đang tinh chỉnh (30 tổ hợp × 5 fold = 150 lần huấn luyện)...


KeyboardInterrupt: 

## Bước 3b: RQ3 — Chuyển giao sang trạm chưa thấy

In [ ]:
from lightgbm import LGBMRegressor, LGBMClassifier

all_stations = feat.station.unique()

random_state = np.random.RandomState(42)
holdout_size = max(3, len(all_stations) // 5)
holdout_stations = random_state.choice(
    all_stations,
    size=holdout_size,
    replace=False,
)

print(f"Số trạm tổng    : {len(all_stations)}")
print(f"Số trạm holdout : {len(holdout_stations)} (chưa từng thấy)")
print(f"Số trạm train   : {len(all_stations) - len(holdout_stations)}")

train_seen_stations = train[~train.station.isin(holdout_stations)]
test_unseen_stations = test[test.station.isin(holdout_stations)]

print(f"\nTrain (trạm đã thấy) : {len(train_seen_stations):,} dòng")
print(f"Test (trạm chưa thấy): {len(test_unseen_stations):,} dòng")

# Chuẩn bị delta target và persistence cho tập seen/unseen
sm8_train_seen = train_seen_stations["sm8"].values
sm8_test_unseen = test_unseen_stations["sm8"].values
y_train_seen_delta = train_seen_stations["y_24h"].values - sm8_train_seen

transfer_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    random_state=0,
    verbose=-1,
).fit(
    train_seen_stations[feature_columns],
    y_train_seen_delta,   # ← DELTA
)

X_unseen = test_unseen_stations[feature_columns]
y_unseen = test_unseen_stations["y_24h"].values

prediction_unseen = transfer_model.predict(X_unseen) + sm8_test_unseen   # ← khôi phục
mae_unseen = MAE(y_unseen, prediction_unseen)

# So với naive trên cùng trạm chưa thấy
naive_prediction_unseen = sm8_test_unseen
mae_naive_unseen = MAE(y_unseen, naive_prediction_unseen)

# So với LightGBM huấn luyện trên toàn bộ trạm — tức mô hình chính
mae_lightgbm_full = MAE(y_te, prediction)

rq3_table = pd.DataFrame([
    ["LightGBM full (baseline)",       mae_lightgbm_full],
    ["LightGBM transfer (unseen)",     mae_unseen],
    ["Naive persistence (unseen)",     mae_naive_unseen],
], columns=["model", "MAE"])

rq3_table.to_csv("report/table_rq3_transfer.csv", index=False)
print("\n" + "=" * 60)
print("BẢNG RQ3 — CHUYỂN GIAO SANG TRẠM CHƯA THẤY")
print("=" * 60)
print(rq3_table.round(3))

# Đánh giá
transfer_gap = mae_unseen - mae_lightgbm_full
if transfer_gap < 0.5:
    print(f"\n✓ Chuyển giao TỐT: chỉ mất {transfer_gap:.3f} MAE so với mô hình full")
else:
    print(f"\n⚠ Chuyển giao YẾU: mất {transfer_gap:.3f} MAE, cần fine-tuning")

Số trạm tổng    : 22
Số trạm holdout : 4 (chưa từng thấy)
Số trạm train   : 18

Train (trạm đã thấy) : 303,633 dòng
Test (trạm chưa thấy): 19,217 dòng

BẢNG RQ3 — CHUYỂN GIAO SANG TRẠM CHƯA THẤY
                        model    MAE
0    LightGBM full (baseline)  0.682
1  LightGBM transfer (unseen)  0.332
2  Naive persistence (unseen)  0.268

✓ Chuyển giao TỐT: chỉ mất -0.350 MAE so với mô hình full


### 3b.2. Ablation — mưa và nhiệt độ đóng góp bao nhiêu?

In [ ]:
weather_features = ["prcp", "r24", "r7d", "tair", "t3d", "ws"]
non_weather_features = [
    c for c in feature_columns
    if c not in weather_features
]

print(f"Feature khí tượng    : {weather_features}")
print(f"Feature giữ lại (16 - 6): {non_weather_features}")

model_no_weather = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    random_state=0,
    verbose=-1,
).fit(
    train[non_weather_features],
    y_tr_delta,   # ← DELTA
)

prediction_no_weather = model_no_weather.predict(test[non_weather_features]) + sm8_te   # ← khôi phục
mae_no_weather = MAE(y_te, prediction_no_weather)

mae_with_weather = MAE(y_te, prediction)   

ablation_table = pd.DataFrame([
    ["LightGBM có khí tượng",     mae_with_weather],
    ["LightGBM không khí tượng", mae_no_weather],
], columns=["model", "MAE"])

ablation_table.to_csv("report/table_rq3_ablation.csv", index=False)
print(ablation_table.round(3))

# Tính đóng góp %
weather_contribution = (mae_no_weather - mae_with_weather) / mae_with_weather * 100
print(f"\nĐóng góp của khí tượng: {weather_contribution:.1f}% giảm MAE")
if weather_contribution > 10:
    print("→ Khí tượng QUAN TRỌNG, giữ trong mô hình")
else:
    print("→ Khí tượng đóng góp ÍT, có thể xét bỏ")

Feature khí tượng    : ['prcp', 'r24', 'r7d', 'tair', 't3d', 'ws']
Feature giữ lại (16 - 6): ['sm2', 'sm4', 'sm8', 'sm20', 'sm8_lag1', 'sm8_lag6', 'sm8_lag24', 'sm8_lag7d', 'sm8_delta6', 'sm8_delta24', 'st2', 'hr', 'mon', 'st_code']
                      model    MAE
0     LightGBM có khí tượng  0.682
1  LightGBM không khí tượng  0.733

Đóng góp của khí tượng: 7.4% giảm MAE
→ Khí tượng đóng góp ÍT, có thể xét bỏ


### 3b.3. Cảnh báo hạn — Recall và Precision

In [ ]:
drought_threshold = 10        # sm8 < 10% = hạn thực tế
alarm_threshold   = 11        # pred < 11% = cảnh báo

truth_drought = y_te < drought_threshold       # boolean array — hạn thực tế
alarm_triggered = prediction < alarm_threshold  # boolean array — có cảnh báo

tp = (alarm_triggered & truth_drought).sum()          # cảnh báo đúng
fp = (alarm_triggered & ~truth_drought).sum()         # cảnh báo nhầm
fn = (~alarm_triggered & truth_drought).sum()         # bỏ sót
tn = (~alarm_triggered & ~truth_drought).sum()        # im lặng đúng

print(f"True Positive (báo đúng)  : {tp:,}")
print(f"False Positive (báo nhầm) : {fp:,}")
print(f"False Negative (bỏ sót)   : {fn:,}")
print(f"True Negative (im lặng)   : {tn:,}")

# max(x, 1) để tránh chia 0 nếu không có hạn thực tế nào
recall = tp / max(truth_drought.sum(), 1)
precision = tp / max(alarm_triggered.sum(), 1)

# F1 = trung bình điều hoà của recall và precision
f1 = 2 * recall * precision / max(recall + precision, 1e-9)

alarm_table = pd.DataFrame([{
    "recall":    round(recall, 3),
    "precision": round(precision, 3),
    "f1":        round(f1, 3),
    "TP": tp, "FP": fp, "FN": fn, "TN": tn,
}])

alarm_table.to_csv("report/table_rq3_drought_alarm.csv", index=False)
print("\n" + "=" * 60)
print("CẢNH BÁO HẠN")
print("=" * 60)
print(f"Recall    (bắt được):  {recall:.2%}")
print(f"Precision (đúng)   :  {precision:.2%}")
print(f"F1                 :  {f1:.2%}")

if recall >= 0.8:
    print("\n✓ Recall > 80% — hệ thống ĐỦ AN TOÀN cho nông dân")
else:
    print(f"\n⚠ Recall chỉ {recall:.0%} — cân nhắc hạ ngưỡng cảnh báo")

True Positive (báo đúng)  : 12,241
False Positive (báo nhầm) : 2,963
False Negative (bỏ sót)   : 278
True Negative (im lặng)   : 79,548

CẢNH BÁO HẠN
Recall    (bắt được):  97.78%
Precision (đúng)   :  80.51%
F1                 :  88.31%

✓ Recall > 80% — hệ thống ĐỦ AN TOÀN cho nông dân


## Bước 3c: Giải thích mô hình bằng SHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_te)   

shap_importance = (
    pd.DataFrame({
        "feature":        feature_columns,
        "mean_abs_shap":  np.abs(shap_values).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance.to_csv("report/table_shap.csv", index=False)
print("BẢNG ĐỘ QUAN TRỌNG FEATURE (theo SHAP)")
print("=" * 60)
print(shap_importance.round(3))

plt.figure()
shap.summary_plot(
    shap_values, X_te,
    feature_names=feature_columns,
    show=False,
)
plt.savefig("report/fig_shap.png", dpi=300, bbox_inches="tight")
plt.close()
print("\nĐã lưu: report/fig_shap.png")

top_3_features = shap_importance["feature"].head(3).tolist()
print(f"\n3 feature quan trọng nhất: {top_3_features}")

for feature_name in top_3_features:
    feature_index = feature_columns.index(feature_name)

    plt.figure()
    shap.dependence_plot(
        feature_index, shap_values, X_te,
        feature_names=feature_columns,
        show=False,
    )
    output_path = f"report/fig_shap_dep_{feature_name}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Đã lưu: {output_path}")

top_5_across_seeds = []

# TRƯỚC: model_seed = make_lightgbm(seed).fit(X_tr, y_tr)
# SAU:
for seed in range(3):
    model_seed = make_lightgbm(seed).fit(X_tr, y_tr_delta)   # ← DELTA
    sv_seed = shap.TreeExplainer(model_seed).shap_values(X_te)
    # ... phần còn lại giữ nguyên

    # LGBMRegressor trả về array, LGBMClassifier trả về list
    if isinstance(sv_seed, list):
        sv_seed = sv_seed[1]

    importance_seed = pd.Series(
        np.abs(sv_seed).mean(axis=0),
        index=feature_columns,
    )
    top_5 = set(importance_seed.nlargest(5).index)
    top_5_across_seeds.append(top_5)

# Đếm số feature chung trong top 5 qua 3 seed
common_features = (
    top_5_across_seeds[0]
    & top_5_across_seeds[1]
    & top_5_across_seeds[2]
)

print(f"\nSố feature chung trong top 5 qua 3 seed: {len(common_features)} / 5")
if len(common_features) >= 4:
    print("→ Top feature ỔN ĐỊNH, giải thích SHAP đáng tin")
else:
    print("→ Top feature KHÔNG ổn định, cần thêm seed hoặc regularize mạnh hơn")

with open("report/params_best.json", "w") as f:
    json.dump(best_model.get_params(), f, indent=2, default=str)
print("\nĐã lưu tham số: report/params_best.json")

BẢNG ĐỘ QUAN TRỌNG FEATURE (theo SHAP)
        feature  mean_abs_shap
0           sm8          0.244
1       st_code          0.177
2           sm2          0.163
3           r24          0.158
4     sm8_lag24          0.129
5   sm8_delta24          0.114
6      sm8_lag1          0.109
7           sm4          0.097
8           t3d          0.084
9           mon          0.082
10         sm20          0.068
11   sm8_delta6          0.066
12     sm8_lag6          0.065
13         prcp          0.061
14         tair          0.059
15    sm8_lag7d          0.057
16          r7d          0.056
17          st2          0.024
18           ws          0.022
19           hr          0.015

Đã lưu: report/fig_shap.png

3 feature quan trọng nhất: ['sm8', 'st_code', 'sm2']
  Đã lưu: report/fig_shap_dep_sm8.png
  Đã lưu: report/fig_shap_dep_st_code.png
  Đã lưu: report/fig_shap_dep_sm2.png

Số feature chung trong top 5 qua 3 seed: 5 / 5
→ Top feature ỔN ĐỊNH, giải thích SHAP đáng tin

Đã lưu tham 

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

## Chẩn đoán mô hình — vì sao tốt, vì sao chưa tốt

In [ ]:
best_improvement = result_full.iloc[1:]["improvement_vs_naive_%"].max()

# Tính y_tr_raw từ train (level target gốc, không phải delta)
y_tr_raw = train["y_24h"].astype("float32").values

# Sửa: train_mae phải cộng sm8_tr để về level
train_prediction = best_model.predict(X_tr) + sm8_tr
train_mae = MAE(y_tr_raw, train_prediction)
test_mae = MAE(y_te, prediction)

test_with_error = test.assign(
    pred=prediction,
    err=np.abs(test["y_24h"] - prediction),
)

# Lấy tháng từ timestamp
if not np.issubdtype(test_with_error["ts"].dtype, np.number):
    test_with_error["month"] = pd.to_datetime(
        test_with_error["ts"], errors="coerce"
    ).dt.month
else:
    test_with_error["month"] = test_with_error["ts"]

error_by_month = test_with_error.groupby("month")["err"].mean()
worst_month = error_by_month.idxmax()
worst_month_mae = error_by_month.max()

top_10_percent = test_with_error.nlargest(
    int(0.1 * len(test_with_error)),
    "y_24h",
)
mae_at_peaks = top_10_percent["err"].mean()

diagnosis_table = pd.DataFrame([
    [
        "Có đánh bại naive?",
        f"{best_improvement:.1f}% cải thiện",
        "TỐT" if best_improvement > 10 else "chưa đủ, xem lại feature",
    ],
    [
        "Có overfitting?",
        f"Train MAE = {train_mae:.2f}, Test MAE = {test_mae:.2f}",
        "CÓ dấu hiệu" if test_mae > 1.5 * train_mae else "KHÔNG có",
    ],
    [
        "Sai theo period?",
        f"Tháng {worst_month} có MAE = {worst_month_mae:.2f}",
        "xem lại feature seasonal",
    ],
    [
        "Sai ở giá trị cực đoan?",
        f"MAE trên top 10% = {mae_at_peaks:.2f}",
        "cân nhắc quantile loss" if mae_at_peaks > 1.5 * test_mae else "OK",
    ],
], columns=["question", "evidence", "assessment"])

diagnosis_table.to_csv("report/table_diagnosis.csv", index=False)
print("BẢNG CHẨN ĐOÁN")
print("=" * 80)
print(diagnosis_table.to_string(index=False))

BẢNG CHẨN ĐOÁN
               question                          evidence               assessment
     Có đánh bại naive?                   13.8% cải thiện                      TỐT
        Có overfitting? Train MAE = 0.46, Test MAE = 0.68                 KHÔNG có
       Sai theo period?             Tháng 4 có MAE = 1.11 xem lại feature seasonal
Sai ở giá trị cực đoan?           MAE trên top 10% = 1.16   cân nhắc quantile loss


## Test cases của Notebook B

In [ ]:
best_improvement = result_full.iloc[1:]["improvement_vs_naive_%"].max()

# Train MAE cũng phải cộng sm8_tr để về level
train_prediction = best_model.predict(X_tr) + sm8_tr
train_mae = MAE(y_tr_raw, train_prediction)
test_mae = MAE(y_te, prediction)

test_with_error = test.assign(
    pred=prediction,
    err=np.abs(test["y_24h"] - prediction),
)

# Lấy tháng từ timestamp
if not np.issubdtype(test_with_error["ts"].dtype, np.number):
    test_with_error["month"] = pd.to_datetime(
        test_with_error["ts"], errors="coerce"
    ).dt.month
else:
    test_with_error["month"] = test_with_error["ts"]

error_by_month = test_with_error.groupby("month")["err"].mean()
worst_month = error_by_month.idxmax()
worst_month_mae = error_by_month.max()

top_10_percent = test_with_error.nlargest(
    int(0.1 * len(test_with_error)),
    "y_24h",
)
mae_at_peaks = top_10_percent["err"].mean()

diagnosis_table = pd.DataFrame([
    [
        "Có đánh bại naive?",
        f"{best_improvement:.1f}% cải thiện",
        "TỐT" if best_improvement > 10 else "chưa đủ, xem lại feature",
    ],
    [
        "Có overfitting?",
        f"Train MAE = {train_mae:.2f}, Test MAE = {test_mae:.2f}",
        "CÓ dấu hiệu" if test_mae > 1.5 * train_mae else "KHÔNG có",
    ],
    [
        "Sai theo period?",
        f"Tháng {worst_month} có MAE = {worst_month_mae:.2f}",
        "xem lại feature seasonal",
    ],
    [
        "Sai ở giá trị cực đoan?",
        f"MAE trên top 10% = {mae_at_peaks:.2f}",
        "cân nhắc quantile loss" if mae_at_peaks > 1.5 * test_mae else "OK",
    ],
], columns=["question", "evidence", "assessment"])

diagnosis_table.to_csv("report/table_diagnosis.csv", index=False)
print("BẢNG CHẨN ĐOÁN")
print("=" * 80)
print(diagnosis_table.to_string(index=False))

BẢNG CHẨN ĐOÁN
               question                          evidence               assessment
     Có đánh bại naive?                   13.8% cải thiện                      TỐT
        Có overfitting? Train MAE = 0.46, Test MAE = 0.68                 KHÔNG có
       Sai theo period?             Tháng 4 có MAE = 1.11 xem lại feature seasonal
Sai ở giá trị cực đoan?           MAE trên top 10% = 1.16   cân nhắc quantile loss
